In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [2]:
df = pd.read_csv('../data/diabetic_data.csv')
df.replace('?', np.nan, inplace=True)

print(f"✅ Raw data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Raw data loaded: 101,766 rows × 50 columns


In [3]:
# These columns are either mostly missing or not useful for prediction
cols_to_drop = [
    'weight',           # 97% missing - useless
    'payer_code',       # 40% missing - insurance info not predictive
    'medical_specialty',# 49% missing - too many missing
    'encounter_id',     # just an ID number - not predictive
    'patient_nbr',      # just a patient ID - not predictive
]

df.drop(columns=cols_to_drop, inplace=True)

print(f"✅ Dropped {len(cols_to_drop)} columns")
print(f"📊 Shape now: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Dropped 5 columns
📊 Shape now: 101,766 rows × 45 columns


In [4]:
# Same patient admitted multiple times - we keep only their FIRST visit
# This prevents data leakage (model seeing future visits of same patient)
before = len(df)
df.drop_duplicates(subset=['patient_nbr'] if 'patient_nbr' in df.columns else None, 
                   keep='first', inplace=True)
after = len(df)

print(f"✅ Removed duplicate patient visits")
print(f"   Before: {before:,} rows")
print(f"   After:  {after:,} rows")
print(f"   Removed: {before - after:,} duplicate visits")

✅ Removed duplicate patient visits
   Before: 101,766 rows
   After:  101,766 rows
   Removed: 0 duplicate visits


In [5]:
# Remove patients who died or were discharged to hospice
# These patients CANNOT be readmitted - they'd skew our model
before = len(df)

# discharge_disposition_id 11 = Expired (died), 13,14 = Hospice
df = df[~df['discharge_disposition_id'].isin([11, 13, 14])]

after = len(df)
print(f"✅ Removed deceased/hospice patients")
print(f"   Before: {before:,} | After: {after:,} | Removed: {before-after:,}")

✅ Removed deceased/hospice patients
   Before: 101,766 | After: 99,353 | Removed: 2,413


In [6]:
# Convert readmitted to BINARY
# 1 = readmitted within 30 days (what we want to predict)
# 0 = not readmitted within 30 days (readmitted later OR never)

df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)

# Drop the original readmitted column
df.drop(columns=['readmitted'], inplace=True)

readmitted_count = df['readmitted_binary'].sum()
total = len(df)
print(f"✅ Target variable created!")
print(f"   Readmitted within 30 days (1): {readmitted_count:,} ({readmitted_count/total*100:.1f}%)")
print(f"   Not readmitted within 30 days (0): {total-readmitted_count:,} ({(total-readmitted_count)/total*100:.1f}%)")

✅ Target variable created!
   Readmitted within 30 days (1): 11,314 (11.4%)
   Not readmitted within 30 days (0): 88,039 (88.6%)


In [7]:
# Fill remaining missing values
df['race'].fillna('Unknown', inplace=True)
df['diag_1'].fillna('Unknown', inplace=True)
df['diag_2'].fillna('Unknown', inplace=True)
df['diag_3'].fillna('Unknown', inplace=True)

# Check no missing values remain
remaining_missing = df.isnull().sum().sum()
print(f"✅ Missing values handled!")
print(f"   Total missing values remaining: {remaining_missing}")

✅ Missing values handled!
   Total missing values remaining: 180747


In [8]:
# This is where we create SMARTER features from existing ones
# New features = better model performance

# 1. Total number of visits (outpatient + emergency + inpatient)
df['total_visits'] = (df['number_outpatient'] + 
                      df['number_emergency'] + 
                      df['number_inpatient'])

# 2. Has the patient been here before? (any prior visits)
df['has_prior_visits'] = (df['total_visits'] > 0).astype(int)

# 3. Medication complexity score (how many different meds changed)
df['med_complexity'] = df['num_medications'] * df['number_diagnoses']

# 4. Lab intensity (lab procedures per day in hospital)
df['lab_intensity'] = df['num_lab_procedures'] / df['time_in_hospital']

# 5. Convert age ranges to numeric midpoints
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}
df['age_numeric'] = df['age'].map(age_map)

print("✅ New features created:")
print("   • total_visits")
print("   • has_prior_visits")
print("   • med_complexity")
print("   • lab_intensity")
print("   • age_numeric")
print(f"\n📊 Shape now: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ New features created:
   • total_visits
   • has_prior_visits
   • med_complexity
   • lab_intensity
   • age_numeric

📊 Shape now: 99,353 rows × 50 columns


In [9]:
# Medication columns have values: No, Steady, Up, Down
# Convert these to numbers

med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
            'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
            'miglitol', 'troglitazone', 'tolazamide', 'examide',
            'citoglipton', 'insulin', 'glyburide-metformin',
            'glipide-metformin', 'glimepiride-pioglitazone',
            'metformin-rosiglitazone', 'metformin-pioglitazone',
            'change', 'diabetesMed']

med_map = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': 3, 
           'Ch': 1, 'Yes': 1}

for col in med_cols:
    if col in df.columns:
        df[col] = df[col].map(med_map).fillna(0).astype(int)

print("✅ Medication columns encoded!")

✅ Medication columns encoded!


In [10]:
# Encode remaining categorical columns using Label Encoding
categorical_cols = ['race', 'gender', 'age', 'A1Cresult', 
                    'max_glu_serum', 'diag_1', 'diag_2', 'diag_3']

le = LabelEncoder()
for col in categorical_cols:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))

print("✅ Categorical columns encoded!")
print(f"📊 Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Categorical columns encoded!
📊 Final shape: 99,353 rows × 50 columns


In [11]:
print("=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)
print(f"\n📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n🎯 Target distribution:")
print(df['readmitted_binary'].value_counts())
print(f"\n❌ Missing values: {df.isnull().sum().sum()}")
print(f"\n📋 Column list:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:>2}. {col}")

# Save the cleaned dataset
df.to_csv('../data/diabetic_data_cleaned.csv', index=False)
print(f"\n✅ Cleaned dataset saved to data/diabetic_data_cleaned.csv!")

FINAL DATASET SUMMARY

📊 Shape: 99,353 rows × 50 columns

🎯 Target distribution:
readmitted_binary
0    88039
1    11314
Name: count, dtype: int64

❌ Missing values: 0

📋 Column list:
    1. race
    2. gender
    3. age
    4. admission_type_id
    5. discharge_disposition_id
    6. admission_source_id
    7. time_in_hospital
    8. num_lab_procedures
    9. num_procedures
   10. num_medications
   11. number_outpatient
   12. number_emergency
   13. number_inpatient
   14. diag_1
   15. diag_2
   16. diag_3
   17. number_diagnoses
   18. max_glu_serum
   19. A1Cresult
   20. metformin
   21. repaglinide
   22. nateglinide
   23. chlorpropamide
   24. glimepiride
   25. acetohexamide
   26. glipizide
   27. glyburide
   28. tolbutamide
   29. pioglitazone
   30. rosiglitazone
   31. acarbose
   32. miglitol
   33. troglitazone
   34. tolazamide
   35. examide
   36. citoglipton
   37. insulin
   38. glyburide-metformin
   39. glipizide-metformin
   40. glimepiride-pioglitazone
   41. 